# Experiment 1 — the first rule against the benchmark

**Steps 4, 5 and 6 of 8 — portfolio, backtest, attribution — for one idea.**

**It produces** a book in `Portfolio/`, a track record in `Backtest/` and a breakdown of the
return in `Attribution/`. **It prevents** a good signal in a portfolio nobody could hold, paper
returns that real trading would have erased, and factor beta sold as alpha.

Experiment 1 is the **first rule tested against the benchmark** `JOURNAL_1.md` chose, and
the yardstick every later experiment is also measured against. It is not a null. It is a real
strategy with a real return, and it can graduate like any other — **its rules freeze once
`FINDINGS_1.md` reports**, because a change to them invalidates every comparison in `RESULTS.md`.
Improvements go into a new experiment, unless the owner rewrites this one, as `AGENTS.md` says.

The hypothesis is in [`BLUEPRINT_1.md`](BLUEPRINT_1.md), **written before this notebook's rule**.
The running log is [`JOURNAL_1.md`](JOURNAL_1.md); the results that survive are in
[`FINDINGS_1.md`](FINDINGS_1.md).

**In the template this notebook holds no code:** each section below says what is expected in it.
Write the cells.

<!-- example: begin -->

**Golden Flow's Experiment 1 — the most-traded KN US Equity Core members in a golden cross,
by traded value.** The rule is [`BLUEPRINT_1.md`](BLUEPRINT_1.md)'s, committed before this
notebook held a line of it: the KN US Equity Core's members with a golden cross at the prior
close, ranked and weighted by 63-day traded value, no weight above 20% and none below 1%,
re-struck when the held set changes, and a name whose prices are about to stop — a delisting,
a corporate event — sold the day before, judged against the KN US Equity Core, `SPY` and the
same rule without the cross, from 2015-01-02 to 2026-06-01. **Every figure comes from the
engine or the attribution library**; this notebook only writes books, asks for their prices
and reads the answers.

<!-- example: end -->

## Position in the pipeline

This notebook is **only the strategy**. The universe and the data are built by earlier stages and
are simply read here:

```
Universe/universe.ipynb   ->  Security_Master.csv
Data/curator.py           ->  Data/Curator/Time_Series/     m_* + c_*
Data/refinery.py          ->  Data/Refinery/Time_Series/    + r_*      <- this notebook reads here
Data/analyzer.ipynb       ->  the measurements the blueprint's predictions came from
        |
        v
experiment_1.ipynb        ->  Portfolio/  ->  Backtest/  ->  Attribution/
```

**What this notebook does not do.** It does not download anything, profile the universe, or compute
a signal. A number about the data itself belongs in the Universe or Data stage — that separation is
what keeps every experiment comparable, because all of them read the identical panel.

**Four modules beside this notebook are shared by every experiment** — `securities_panel.py`,
`portfolio_construction.py`, `backtest_engine.py` and `attribution_analysis.py`, one per Lab
library — and no strategy column is named in any of them. Experiment 1 loads the panel through
`securities_panel.py` like every later experiment, so the comparison is on the rule and nothing
else.

## The section contract

Every experiment notebook has the same shape, so anyone who has read one can read all of them.
**Everything below section 2 is strategy-agnostic**, given the three objects that section produces.

| Section | Contains |
| --- | --- |
| 0 · Setup | paths, and **the strategy's columns** — the only strategy names in this notebook outside the rule |
| 1 · The panel | load the refined files and reshape them |
| 2 · The rule | selection, sizing, timing. **The one cell you write** |
| 2.1 · Invariants | what every rule must pass, whatever it is |
| 3 · Construction | the book, and the diagnostics a person would run it on |
| 4 · Backtest | one engine pass, guarded import, reports-and-skips without a licence |
| 5 · Attribution | where the return came from, guarded the same way |
| 6 · Counterfactuals | the arms that price who earned the idiosyncratic share |
| 7 · Verdict | what it concluded, **in words** |
| Handoff | what the next stage consumes, and what this one left open |
| 8 · Verify | assertions that raise when the output is wrong: the invariants, the weight file read back, the days each engine run valued against its window's trading days, and the first cut's benchmark within a point a year of the index's own returns file |

## 0 · Setup

Paths, and the strategy's columns. **Declare them here, not in a shared module**, so a signal never
becomes every later experiment's default without anyone deciding it. Read only the columns the
strategy consumes.

**And the exclusions**: the names the blocking rows of `Universe/Data_Issues.csv` list, read from
the register and never typed. A blocking row is a label until a stage reads it; this is the stage.

Three price columns do **three different jobs**, and getting them out of step is silent — the
backtest P&L and the attribution would quietly run on different bases:

| Role | Basis | Why |
| --- | --- | --- |
| Daily mark | dividend-and-split adjusted close | total-return valuation between rebalances |
| Fill | dividend-and-split adjusted VWAP | the price a trade actually gets |
| Commission | **unadjusted** VWAP | per-share cents ride on the unadjusted share count |

A provider may return its VWAP columns as null, which is why the Curator reconstructs both VWAPs as
`c_*`, and nothing below reads a provider's own VWAP column, whichever provider `Data/curator.py`
asks for.

In [ ]:
# EXAMPLE-ONLY CELL
import os
import pathlib
import sys

import numpy
import pandas

# Every path below is relative to the repository root, whichever folder the notebook started in.
NOTEBOOK_DIRECTORY = pathlib.Path.cwd()
REPOSITORY_ROOT = next(
    folder
    for folder in (NOTEBOOK_DIRECTORY, *NOTEBOOK_DIRECTORY.parents)
    if (folder / "Experiments").is_dir() and (folder / "Universe").is_dir()
)
os.chdir(REPOSITORY_ROOT)
sys.path.insert(0, str(REPOSITORY_ROOT / "Experiments"))
sys.path.insert(0, str(REPOSITORY_ROOT / "Data"))

import attribution_analysis  # noqa: E402 - the paths above have to exist first
import backtest_engine  # noqa: E402
import hand_supplied  # noqa: E402
import portfolio_construction  # noqa: E402
import securities_panel  # noqa: E402

EXPERIMENT_DIRECTORY = REPOSITORY_ROOT / "Experiments" / "Experiment_1"

# The window, the bounds and the columns, as BLUEPRINT_1.md fixed them.
WINDOW_START = pandas.Timestamp("2015-01-02")
WINDOW_END = pandas.Timestamp("2026-06-01")
SUB_PERIODS = (
    ("2015 to 2018", WINDOW_START, pandas.Timestamp("2018-12-31")),
    ("2019 to 2022", pandas.Timestamp("2019-01-01"), pandas.Timestamp("2022-12-31")),
    ("2023 to 2026", pandas.Timestamp("2023-01-01"), WINDOW_END),
)
MAXIMUM_WEIGHT = 0.20
MINIMUM_WEIGHT = 0.01
TREND_COLUMN = "r_trend_50_200"
SCORE_COLUMN = "r_traded_value_sma_63d"
MEMBERSHIP_COLUMN = "r_index_weight"
FILL_COLUMN = "c_vwap_dividend_and_split_adjusted"
MARK_COLUMN = "m_close_dividend_and_split_adjusted"
# The blueprint's exclusions but `NE`, the symbol FMP prices the index's `NEBLQ` under, which
# this copy's seed leaves out with every listing whose two tickers differ.
EXPECTED_EXCLUSIONS = ("MNKKQ", "PCP", "RAI")

# The costs every verdict is read at, and the realistic row beside them.
INITIAL_CAPITAL = 1_000_000
COMMISSION_CENTS = 0.05
REALISTIC_COMMISSION_CENTS = 0.006
SLIPPAGE_BASIS_POINTS = 5.0
CASH_RESERVE = 0.01

# The sweep, one setting at a time; the counterfactual seeds; the parallel workers.
SWEEP = (
    ("cap 15%", {"maximum_weight": 0.15}),
    ("cap 25%", {"maximum_weight": 0.25}),
    ("floor 0.5%", {"minimum_weight": 0.005}),
    ("floor 2%", {"minimum_weight": 0.02}),
    ("traded value over 21 days", {"score_column": "r_traded_value_sma_21d"}),
    ("traded value over 126 days", {"score_column": "r_traded_value_sma_126d"}),
    ("cross 20/100", {"trend_column": "r_trend_20_100"}),
    ("cross 100/300", {"trend_column": "r_trend_100_300"}),
)
RANDOM_SEEDS = tuple(range(1, 21))
WORKERS = min(10, os.cpu_count() or 1)

# The window is the universe notebook's, and the exclusions are its register's blocking rows: the
# blueprint fixed both, and a run on anything else is not this experiment.
window_written = pandas.read_csv("Universe/Window.csv")
issues = pandas.read_csv("Universe/Data_Issues.csv").fillna("")
blocking = issues[
    (issues["severity"] == "blocking") & ~issues["check"].str.startswith("missing file")
]
EXCLUDED = tuple(sorted({
    identifier
    for identifiers in blocking["identifiers"]
    for identifier in str(identifiers).split()
}))
print(f"window {WINDOW_START.date()} to {WINDOW_END.date()}; the universe notebook's: "
      f"{window_written['window_start'].iloc[0]} to {window_written['window_end'].iloc[0]}")
print(f"excluded by name: {', '.join(EXCLUDED)}")
print(f"engine installed: {backtest_engine.ENGINE_INSTALLED}; "
      f"attribution installed: {attribution_analysis.LIBRARY_INSTALLED}")

## 1 · The panel

Load the refined files, resolve **one position per security**, and reshape to matrices.

A point-in-time universe contains renamed securities: two identifiers sharing one identity, each
carrying part of the history. Left alone they are two independent positions and the book
double-counts at the changeover. Key positions by a stable identity — an ISIN where the seed
carries one — falling back to the identifier itself, and where two legs overlap on a date let the
leg still reporting later win.

The long panel then becomes one wide `dates x securities` matrix per input, which is what makes the
whole rule in section 2 a handful of vectorised lines instead of a loop over files.

In [ ]:
# EXAMPLE-ONLY CELL
COLUMNS = (
    TREND_COLUMN,
    "r_trend_20_100",
    "r_trend_100_300",
    SCORE_COLUMN,
    "r_traded_value_sma_21d",
    "r_traded_value_sma_126d",
    MEMBERSHIP_COLUMN,
    FILL_COLUMN,
    MARK_COLUMN,
)
matrices = securities_panel.load_matrices(COLUMNS)
# Nothing after the window's last day is read by any cell below.
cut = {
    name: matrix.loc[:WINDOW_END]
    for name, matrix in matrices.items()
}
membership = cut[MEMBERSHIP_COLUMN].fillna(0.0) > 0
priced = cut[FILL_COLUMN].notna()
# A name whose prices stop tomorrow -- delisted, or a corporate event that leaves a gap -- is sold
# today: the rule re-strikes at t-1, at a real price, the owner's instruction of 2026-10-06.
tradable = portfolio_construction.exit_before_price_stops(priced)
allowed = pandas.Series(
    ~cut[FILL_COLUMN].columns.isin(EXCLUDED),
    index=cut[FILL_COLUMN].columns,
)
calendar = cut[FILL_COLUMN].index
window_days = calendar[(calendar >= WINDOW_START) & (calendar <= WINDOW_END)]
print(f"panel: {len(calendar)} dates x {len(allowed)} positions; window: {len(window_days)} days")
print(f"members per window day: {int(membership.loc[window_days].sum(axis=1).min())} "
      f"to {int(membership.loc[window_days].sum(axis=1).max())}")

## 2 · The rule — the one cell you write

Three statements, in order: **who is eligible**, **how much of each**, and **when to trade**.

**The contract this cell must satisfy** — everything below reads exactly these three objects:

| Object | Type | Meaning |
| --- | --- | --- |
| `selected_matrix` | `dates x securities` boolean | what the book holds on each day |
| `REBALANCE_DATES` | a date index | the days the book is re-struck |
| `target_weights` | `REBALANCE_DATES x securities` float, rows summing to **at most** 1.0 | the book on each of those days |

**Rows sum to at most one, not to exactly one.** A book that must be fully invested cannot express
a defensive strategy. The residual becomes cash in section 3.1, parked in a real priced instrument,
because the engine's weight file has no cash row of its own.

**Sizing is a seam, not a decision buried in the rule.** Hand the eligible set and a returns
history that has already been cut off before today to a weighting function in
`portfolio_construction.py`, and swapping equal weight for inverse volatility, hierarchical risk
parity or any other method of the KaxaNuk Portfolio Construction library is one line — the module
builds the library's method on that cut history, one rebalance date at a time. That is what makes
two experiments comparable rather than merely adjacent.

**Trade only when something changed.** A signal that has not moved is not a reason to pay
commission.

### Two look-aheads, both stated plainly

**The lag.** The eligible set used on rebalance date *t* is the one observed at *t-1*, and the fill
happens at *t*'s price — a full day between the signal and the fill.

**The delisting exit.** A security that delists must be sold on the **last day it still has a fill
price**, and knowing that day is its last requires seeing the next one. This is the standard
backtest compromise — the alternative, carrying a position that can never be exited, is a larger
distortion — and it is implemented by making a name ineligible on that final day, so the set
changes, the rebalance fires, and the position is sold while a price still exists.

In [ ]:
# EXAMPLE-ONLY CELL
# The rule, from BLUEPRINT_1.md. Who may be held on day t: a member at the close of t-1 with a
# 200-day history and the golden cross at 1 then, a traded value then, and a fill price on t.
# The control keeps everything but the cross's sign.


def eligibility(
    trend_column: str,
    use_cross: bool,
) -> "pandas.DataFrame":
    """Who may be held on each window day, decided at the prior close."""
    trend_before = cut[trend_column].shift(1)
    member_before = membership.shift(1, fill_value=False)
    eligible = member_before & trend_before.notna() & tradable & allowed

    if use_cross:
        eligible = eligible & (trend_before > 0)

    return eligible.loc[window_days]


def daily_book(
    trend_column: str = TREND_COLUMN,
    score_column: str = SCORE_COLUMN,
    maximum_weight: float = MAXIMUM_WEIGHT,
    minimum_weight: float = MINIMUM_WEIGHT,
    use_cross: bool = True,
) -> "pandas.DataFrame":
    """The target book on every window day; the bounds set the count, the seam cuts the score."""
    return portfolio_construction.bounded_book(
        eligibility(trend_column, use_cross),
        cut[score_column],
        maximum_weight,
        minimum_weight,
    )


def strike(
    book: "pandas.DataFrame",
) -> "pandas.DataFrame":
    """The book on the days its held set changes, which are the only days it trades."""
    dates = portfolio_construction.rebalance_dates_on_change(book > 0)

    return book.loc[dates]


book_daily = daily_book()
target_weights = strike(book_daily)
REBALANCE_DATES = target_weights.index
selected_matrix = (target_weights > 0).reindex(window_days).ffill().fillna(False).astype(bool)
# The control: the same rule without the cross's sign, re-struck only on the rule's own dates.
control_weights = daily_book(use_cross=False).loc[REBALANCE_DATES]
print(f"rebalance dates: {len(REBALANCE_DATES)} of {len(window_days)} window days")
print(f"holdings per rebalance: {int((target_weights > 0).sum(axis=1).min())} to "
      f"{int((target_weights > 0).sum(axis=1).max())}, median "
      f"{float((target_weights > 0).sum(axis=1).median()):.0f}")

## 2.1 · Invariants

Cheap to check here, expensive to discover inside a P&L. **Every rule must pass these unchanged**,
whatever the strategy is:

- no book is more than fully invested, and none is negatively invested;
- no negative weights, if the strategy is long-only;
- **every security paid for today had its signal on at the prior close** — check the signal itself,
  not the composed eligibility, because the signal is the thing that had to exist in advance;
- every security bought is tradable on the day it is bought, so a fill price exists;
- nothing is still held on a day after it stopped being tradable.

In [ ]:
# EXAMPLE-ONLY CELL
# Cheap here, expensive inside a P&L. Every rule must pass these; section 8 raises on them.
invested = target_weights.sum(axis=1)
held = target_weights > 0
trend_yesterday = cut[TREND_COLUMN].shift(1).reindex_like(held)
member_yesterday = membership.shift(1, fill_value=False).reindex_like(held)
priced_today = priced.reindex_like(held)
priced_tomorrow = priced.shift(-1, fill_value=True).reindex_like(held)
invariants = {
    "no book more than fully invested": bool((invested <= 1.0 + 1e-9).all()),
    "no book negatively invested": bool((invested >= -1e-9).all()),
    "no negative weights": bool((target_weights >= -1e-12).all().all()),
    "no weight above the cap": bool((target_weights <= MAXIMUM_WEIGHT + 1e-9).all().all()),
    "no held weight below the floor": bool(
        (~held | (target_weights >= MINIMUM_WEIGHT - 1e-9)).all().all()
    ),
    # The signal itself, not the composed eligibility: the signal is what had to exist in advance.
    "every holding had its cross at 1 at the prior close": bool(
        (~held | (trend_yesterday > 0)).all().all()
    ),
    "every holding was a member at the prior close": bool((~held | member_yesterday).all().all()),
    "every holding has a fill price on the day it is struck": bool(
        (~held | priced_today).all().all()
    ),
    "no holding is left unpriced the next day: a stop is sold at t-1": bool(
        (~held | priced_tomorrow).all().all()
    ),
    "no excluded name is ever held": not bool(
        held[[name for name in EXCLUDED if name in held]].any().any()
    ),
    "the control trades only on the rule's dates": bool(
        control_weights.index.equals(REBALANCE_DATES)
    ),
    "the control never needs the cross, and never lacks the history": bool(
        (~(control_weights > 0) | trend_yesterday.reindex_like(control_weights).notna()).all().all()
    ),
}

for name, passed in invariants.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")

## 3 · Construction — is this a book you would actually run?

**This is where step 4, Portfolio Construction, lives.** Four properties, each with a failure mode
a performance chart would hide:

| Property | What a bad value would mean |
| --- | --- |
| Invested share over time | the eligibility column is not doing what the analyzer says it does |
| Trigger frequency and turnover | the rule fires so often that this is a transaction-cost question, not an alpha one |
| Holdings and concentration | a "diversified" label on a book that is one or two positions |
| Group drift | the strategy is a disguised bet on one group rather than a rotation between them |

Measure turnover **target-to-target**. The realised figure is lower, because between rebalances the
winners drift up on their own; that calculation needs drifted weights and belongs to the backtest.

> **The blueprint's predictions about the *shape* of the book, rather than about its return, are
> settled here — before any backtest.** They are the first ones that can be wrong, and the cheapest
> to be wrong about.

## 3.1 · Write the deliverables

Two views of the same book, because two readers need it: a **long, human-readable** one with names
and classifications attached, and a **wide, identifier-keyed** `portfolio_weights.csv` — the
backtest engine's input, which looks each identifier up in the market-data folder and so has to
speak in identifiers, not in stitched positions.

**This is where cash becomes a position.** Everything above lets a book be less than fully
invested; here the residual becomes a weight in the cash proxy, so the engine charges commission on
going to cash and earns the yield while there. A strategy whose defining move is *sell everything*
has to pay for it.

In [ ]:
# EXAMPLE-ONLY CELL
# Is this a book you would actually run? The target weights held forward between trade dates --
# the drift is the engine's -- so this is the book's shape, not its return.
years = (WINDOW_END - WINDOW_START).days / 365.25
held_forward = target_weights.reindex(window_days).ffill().fillna(0.0)
holdings_count = (held_forward > 0).sum(axis=1)
effective_positions = 1 / held_forward.pow(2).sum(axis=1).replace(0, numpy.nan)
turnover = target_weights.diff().abs().sum(axis=1) / 2
turnover.iloc[0] = target_weights.iloc[0].sum() / 2
cap_binding = (target_weights >= MAXIMUM_WEIGHT - 1e-9).any(axis=1)
construction = pandas.Series({
    "rebalance dates": len(REBALANCE_DATES),
    "share of window days re-struck": round(len(REBALANCE_DATES) / len(window_days), 3),
    "holdings, median": float(holdings_count.median()),
    "holdings, minimum": int(holdings_count.min()),
    "holdings, maximum": int(holdings_count.max()),
    "effective positions, mean": round(float(effective_positions.mean()), 1),
    "largest weight, mean of daily maxima": round(float(held_forward.max(axis=1).mean()), 3),
    "rebalance dates the cap binds": int(cap_binding.sum()),
    "share of rebalance dates the cap binds": round(float(cap_binding.mean()), 3),
    "mean invested share": round(float(held_forward.sum(axis=1).mean()), 3),
    "lowest invested share": round(float(held_forward.sum(axis=1).min()), 3),
    "annual turnover, one way, target to target": round(float(turnover.sum() / years), 2),
})
print(construction.to_string())
cap_by_year = cap_binding.groupby(cap_binding.index.year).sum()
print("\nrebalance dates the cap binds, by year:")
print(cap_by_year.to_string())
holdings_by_year = holdings_count.groupby(holdings_count.index.year).median()
print("\nmedian holdings, by year:")
print(holdings_by_year.to_string())

# Capacity, gate criterion 4: every trade's size against the name's 63-day traded value at the
# prior close. The largest book the rule can run while no trade takes more than a share of that
# value is the share times the value, over the weight traded. A bound, not an impact model.
traded_weight = target_weights.diff().abs()
traded_weight.iloc[0] = target_weights.iloc[0]
traded_value_before = cut[SCORE_COLUMN].shift(1).reindex(
    index=target_weights.index,
    columns=target_weights.columns,
)
room = traded_value_before / traded_weight.where(traded_weight > 1e-9)
capacity_rows = {}

for participation in (0.01, 0.05):
    trade_capacity = (room * participation).stack().dropna()
    capacity_rows[f"{participation:.0%} of the name's 63-day traded value"] = {
        "worst trade": float(trade_capacity.min()),
        "1st percentile of trades": float(trade_capacity.quantile(0.01)),
        "median trade": float(trade_capacity.median()),
    }

capacity = pandas.DataFrame(capacity_rows).T
print("\ncapacity -- the largest book, in dollars, at which a trade stays within the share:")
print(capacity.map(lambda value: f"{value:,.0f}").to_string())

master = pandas.read_csv("Universe/Security_Master.csv").set_index("main_identifier")
sector_of = master["sector"].reindex(held_forward.columns).fillna("unknown")
by_sector = held_forward.T.groupby(sector_of).sum().T
yearly_sector = by_sector.groupby(by_sector.index.year).mean()
top_sectors = yearly_sector.mean().sort_values(ascending=False).head(6).index
print("\naverage weight by sector (today's labels, indicative), by year:")
print(yearly_sector[top_sectors].round(3).to_string())

In [ ]:
# EXAMPLE-ONLY CELL
# Every book this experiment prices, written before any is priced: the rule and its control, the
# sweep with a control for each setting on that setting's own dates, the equal-weight arm and the
# random books. Each is a weight file the engine reads, and each is a trial.
pool = (
    membership.shift(1, fill_value=False)
    & tradable
    & cut[SCORE_COLUMN].shift(1).notna()
    & allowed
).loc[window_days]
score_before = cut[SCORE_COLUMN].shift(1)


def random_book(
    seed: int,
) -> "pandas.DataFrame":
    """As many members as the rule holds, drawn at random; as many redrawn as entered its book."""
    generator = numpy.random.default_rng(seed)
    rows = {}
    held_names = []
    previous_rule = set()

    for date in REBALANCE_DATES:
        rule_names = set(target_weights.columns[target_weights.loc[date] > 0])
        entrants = len(rule_names - previous_rule)
        available = list(pool.columns[pool.loc[date].to_numpy()])
        still_available = [name for name in held_names if name in set(available)]
        generator.shuffle(still_available)
        keep = max(0, min(len(still_available), len(rule_names) - entrants))
        kept = still_available[:keep]
        candidates = sorted(set(available) - set(kept))
        drawn = list(generator.choice(candidates, len(rule_names) - len(kept), replace=False))
        held_names = sorted(kept + drawn)
        rows[date] = portfolio_construction.bounded_by_score(
            score_before.loc[date, held_names],
            MAXIMUM_WEIGHT,
            0.0,
        )
        previous_rule = rule_names

    frame = pandas.DataFrame.from_dict(rows, orient="index")

    return frame.reindex(columns=target_weights.columns).fillna(0.0)


equal_weight = (target_weights > 0).astype(float)
equal_weight = equal_weight.div(equal_weight.sum(axis=1), axis=0).clip(upper=MAXIMUM_WEIGHT)
books = {
    "rule": target_weights,
    "control": control_weights,
    "equal weight": equal_weight,
}

for setting, arguments in SWEEP:
    variant = strike(daily_book(**arguments))
    variant_control = daily_book(**arguments, use_cross=False).loc[variant.index]
    books[f"sweep, {setting}"] = variant
    books[f"sweep, {setting}, control"] = variant_control

for seed in RANDOM_SEEDS:
    books[f"random {seed:02d}"] = random_book(seed)

file_names = {
    label: "portfolio_weights_" + label.replace(", ", "_").replace(" ", "_").replace("%", "pct")
    .replace("/", "_").replace(".", "_")
    for label in books
}
file_names["rule"] = "portfolio_weights"

for label, weights in books.items():
    backtest_engine.write_weight_file(
        securities_panel.expand_to_identifiers(weights),
        EXPERIMENT_DIRECTORY,
        file_names[label],
    )

print(f"books written: {len(books)}")
print(pandas.Series({label: len(weights) for label, weights in books.items()}).to_string())

## 4 · Backtest — KaxaNuk Backtest Engine

**In plain words:** run the rules over history, with costs, without peeking ahead.

The weight file goes to the licensed engine, which simulates the book share by share: it fills at a
real price, charges per-share commission on the unadjusted price, holds integer share counts and a
cash reserve, marks the portfolio daily between rebalances, and compares against the benchmarks.

**This is the only backtest in the repository**, and results are accepted **net** or not at all.
Clip the window to the shortest benchmark up front rather than discovering it as a crash, and report
which benchmark bound it.

> **Guard the import.** The engine installs from KaxaNuk's licensed index rather than PyPI, so this
> section reports what is missing and skips without it. Everything in `Portfolio/` is already
> written and does not depend on the engine — a clone with no licence gets a real book and no
> numbers, by design.

In [ ]:
# EXAMPLE-ONLY CELL
# Priced by the engine, in parallel, each run cached under the hash of its weight file. The cost
# rows and the sub-periods re-price the rule and its control; they add no book.


def task(
    portfolio_name: str,
    start: "pandas.Timestamp",
    end: "pandas.Timestamp",
    commission_cents: float = COMMISSION_CENTS,
) -> dict:
    """One engine run's settings."""
    return {
        "experiment_directory": str(EXPERIMENT_DIRECTORY),
        "portfolio_name": portfolio_name,
        "start": start.date().isoformat(),
        "end": end.date().isoformat(),
        "initial_capital": INITIAL_CAPITAL,
        "commission_cents": commission_cents,
        "slippage_basis_points": SLIPPAGE_BASIS_POINTS,
        "cash_reserve_percentage": CASH_RESERVE,
    }


def from_start(
    weights: "pandas.DataFrame",
    start: "pandas.Timestamp",
    end: "pandas.Timestamp",
) -> "pandas.DataFrame":
    """The book struck again on a sub-period's first day, then on its own dates inside it."""
    first_day = window_days[window_days >= start][0]
    in_force = weights.loc[:first_day].iloc[-1]
    inside = weights.loc[(weights.index > first_day) & (weights.index <= end)]
    opening = pandas.DataFrame([in_force], index=[first_day])

    return pandas.concat([opening, inside])


tasks = [
    task(file_names[label], WINDOW_START, WINDOW_END)
    for label in books
]
runs_by_label = {
    label: file_names[label]
    for label in books
}

for label in ("rule", "control"):
    realistic_name = f"{file_names[label]}_realistic"
    backtest_engine.write_weight_file(
        securities_panel.expand_to_identifiers(books[label]),
        EXPERIMENT_DIRECTORY,
        realistic_name,
    )
    tasks.append(task(realistic_name, WINDOW_START, WINDOW_END, REALISTIC_COMMISSION_CENTS))
    runs_by_label[f"{label}, realistic commission"] = realistic_name

    for period, start, end in SUB_PERIODS:
        period_name = f"{file_names[label]}_{period.replace(' ', '_')}"
        backtest_engine.write_weight_file(
            securities_panel.expand_to_identifiers(from_start(books[label], start, end)),
            EXPERIMENT_DIRECTORY,
            period_name,
        )
        tasks.append(task(period_name, start, end))
        runs_by_label[f"{label}, {period}"] = period_name

if backtest_engine.ENGINE_INSTALLED:
    priced = backtest_engine.price_books(tasks, WORKERS)
    runs = {
        label: priced[name]
        for label, name in runs_by_label.items()
    }
    failed = {
        label: run["error"]
        for label, run in runs.items()
        if not run["success"]
    }
    print(f"engine runs: {len(runs)}, failed: {len(failed)}")

    for label, error in failed.items():
        print(f"  {label}: {error}")
else:
    runs = {}

# Step 5 stands on the rule's own run: without the engine, or with a licence that does not
# validate, which the engine returns as a failed run, every cell that reads a run skips.
rule_run = runs.get("rule", {"error": "the KaxaNuk Backtest Engine is not installed"})
ENGINE_READY = bool(rule_run.get("success", False))

if not ENGINE_READY:
    print(f"step 5 skipped: {rule_run['error']}")

In [ ]:
# EXAMPLE-ONLY CELL
# Every figure below comes from the engine. There is no second simulator in this repository.
STATISTICS = {
    "CAGR": "Annualized Return (CAGR)",
    "volatility": "Annualized Volatility",
    "Sharpe": "Portfolio Sharpe Ratio",
    "Sortino": "Portfolio Sortino Ratio",
    "max drawdown": "Max Drawdown",
}

if ENGINE_READY:
    rows = {}

    for label, run in runs.items():
        if not run["success"]:
            continue

        statistics = run["portfolio_stats"]
        rows[label] = {name: statistics.get(key) for name, key in STATISTICS.items()}
        rows[label]["alpha vs the index"] = statistics.get("Alpha")
        rows[label]["information ratio"] = statistics.get("Information Ratio")
        rows[label]["commissions"] = statistics.get("Total Commissions")
        rows[label]["slippage"] = statistics.get("Total Slippage Costs")
        rows[label]["window"] = run["window"]

    summary = pandas.DataFrame(rows).T
    benchmark_rows = {}

    for label in ("rule", "rule, 2015 to 2018", "rule, 2019 to 2022", "rule, 2023 to 2026"):
        for identifier, statistics in runs[label]["benchmarks_stats"].items():
            benchmark_rows[f"{identifier}, as priced beside {label}"] = {
                name: statistics.get(key) for name, key in STATISTICS.items()
            }

    benchmarks = pandas.DataFrame(benchmark_rows).T
    headline = ["rule", "control", "equal weight", "rule, realistic commission",
                "control, realistic commission"]
    print(summary.loc[headline, list(STATISTICS)].astype(float).round(4).to_string())
    print()
    print(benchmarks.astype(float).round(4).to_string())
    print()
    print(summary.loc[headline, ["alpha vs the index", "information ratio", "commissions",
                                 "slippage", "window"]].to_string())
    EXPERIMENT_DIRECTORY.joinpath("Backtest").mkdir(exist_ok=True)
    summary.to_csv(EXPERIMENT_DIRECTORY / "Backtest" / "summary.csv")
    benchmarks.to_csv(EXPERIMENT_DIRECTORY / "Backtest" / "benchmarks.csv")

## 5 · Attribution — KaxaNuk Attribution Analysis

**In plain words:** which part of the return did you actually earn?

The backtest says *how much* the book made; attribution says **where it came from**:

- **Brinson-Fachler**, the first cut: active return into an **allocation** effect — being
  overweight the right groups — and a **selection** effect, picking the right securities inside
  them. The exact lever that moved.
- **A factor model**, the second layer: excess return into **compensated factor tilts** — beta,
  momentum, residual volatility, liquidity — and **idiosyncratic** alpha, what was earned on
  purpose rather than by accident.
- **Brinson-Fachler again, on the residual**, the third pass: the selection story sharpens, and
  it says whether the Sharpe survives once the factor turns.

**What it settles and what it does not** is in [`../../AGENTS.md`](../../AGENTS.md) — including the
four counterfactual books that answer what the factor model cannot.

**Two inputs come from KaxaNuk's Analytics Factory**, read by `Data/hand_supplied.py` from the
folder `KN_ANALYTICS_PATH` names, or from `Data/Curator/Benchmarks/` and `Data/Curator/Factors/` —
the benchmark's weights and returns, and the factor returns. No price provider sells them. Getting
their layout wrong makes the loader read the attribution transposed rather than fail, so shape them
in one place and say what is missing before trying.

**The book arrives daily.** The attribution library rejects a weight file that is not a daily
series, so it reads the book as the engine held it each trading day, drift included, from
`Backtest/` — never `portfolio_weights.csv`, which holds only the rebalance dates.

**Every weight is held overnight.** The engine's daily book and the index's holdings are struck
at a day's close, after that day's return has moved them, and the library pairs a weight with the
return of its own date. Paired as they arrive, every book — the index's too — earns again the
day's move its weights already hold, several points a year of it. Both tables move one day on
before any pass, so the close of t−1 earns day t, and section 8 raises unless the first cut's
benchmark comes within a point a year of the index's own returns file.

**What binds the window.** The attribution period is the intersection of the factor files and the
benchmark holdings, so it is usually *shorter* than the backtest. The two sets of numbers describe
different periods and must not be compared directly. Record both windows in `FINDINGS_1.md`.

In [ ]:
# EXAMPLE-ONLY CELL
missing = attribution_analysis.report_missing_inputs()
ATTRIBUTION_READY = ENGINE_READY and not missing

if not ATTRIBUTION_READY:
    print("step 6 skipped:", "; ".join(missing) or "step 5 was skipped")
else:
    market_factor = pandas.read_csv(hand_supplied.factor_directory() / "Market.csv", usecols=[0])
    factor_dates = pandas.DatetimeIndex(pandas.to_datetime(market_factor.iloc[:, 0]))
    window = window_days[(window_days >= factor_dates.min()) & (window_days <= factor_dates.max())]
    members_ever = membership.loc[window].columns[membership.loc[window].any(axis=0)]
    print(f"backtest window:    {window_days.min().date()} to {window_days.max().date()}")
    print(f"attribution window: {window.min().date()} to {window.max().date()}")


def attribution_task(
    label: str,
    layers: tuple[str, ...],
) -> dict:
    """One book's inputs for the library: widened to the index, every name priced, no nulls."""
    daily_weights = runs[label]["daily_weights"].reindex(window).fillna(0.0)
    book = attribution_analysis.widen_to_benchmark(
        daily_weights,
        index_weights,
        backtest_engine.BENCHMARK_IDENTIFIERS,
    ).drop(columns=list(EXCLUDED), errors="ignore")
    benchmark = index_weights.reindex(columns=book.columns).fillna(0.0)
    # Both tables are struck at a day's close; the close of t-1 is what earns day t.
    book_overnight = attribution_analysis.held_overnight(book)
    benchmark_overnight = attribution_analysis.held_overnight(benchmark)

    return {
        "name": label,
        "book": book_overnight,
        "benchmark": benchmark_overnight,
        "asset_returns": all_returns.reindex(
            index=book_overnight.index,
            columns=book_overnight.columns,
        ).fillna(0.0),
        "layers": layers,
        "cache_directory": str(EXPERIMENT_DIRECTORY / "Attribution" / "cache"),
    }


if ATTRIBUTION_READY:
    # Prices once, for every name any book or the index holds. A blocking row of the register
    # applies to every stage: the excluded names leave the index too, and it is renormalised.
    attributed_labels = ["rule", "control", "equal weight"] + [
        f"random {seed:02d}" for seed in RANDOM_SEEDS
    ]
    names_held = set(members_ever)

    for label in attributed_labels:
        names_held |= set(runs[label]["daily_weights"].columns)

    all_prices = attribution_analysis.load_prices(tuple(sorted(names_held)), window)
    all_returns = attribution_analysis.load_asset_returns(all_prices)
    priced_index = attribution_analysis.load_benchmark_weights(window, all_prices).drop(
        columns=list(EXCLUDED),
        errors="ignore",
    )
    index_weights = priced_index.div(priced_index.sum(axis=1), axis=0).fillna(0.0)
    FULL = ("brinson", "factor", "residual")
    attribution_tasks = [
        attribution_task(label, FULL)
        for label in ("rule", "control", "equal weight")
    ] + [
        attribution_task(f"random {seed:02d}", ("factor",))
        for seed in RANDOM_SEEDS
        if runs[f"random {seed:02d}"]["success"]
    ]
    coverage = attribution_analysis.benchmark_coverage(window, all_prices)
    print("share of the whole index the first cut can price, by year:")
    print(coverage.groupby(coverage.index.year).mean().round(3).to_string())
    attributed = attribution_analysis.attribute_books(attribution_tasks, 6)
    errors = {name: answer["error"] for name, answer in attributed.items() if "error" in answer}
    print(f"books attributed: {len(attributed)}, failed: {len(errors)} {errors}")

In [ ]:
# EXAMPLE-ONLY CELL
# First cut: Brinson-Fachler, per asset, as the library computes it. Then the reconciliation the
# blueprint makes a precondition: the first cut's benchmark against the engine's index.
if ATTRIBUTION_READY:
    rule_answer = attributed["rule"]
    brinson_totals = rule_answer["brinson_totals"]
    print("Brinson-Fachler on the rule, summed over the window, in points:")
    print(brinson_totals.round(2).to_string())
    brinson_daily = rule_answer["brinson_daily"]
    benchmark_columns = [
        column
        for column in brinson_daily.columns
        if "bench" in column.lower() and "return" in column.lower()
    ]
    print(f"\nthe library's daily columns: {list(brinson_daily.columns)}")
    index_returns = hand_supplied.read_benchmark_returns().reindex(window[1:])

    if len(benchmark_columns) > 0:
        first_cut_annual = float(brinson_daily[benchmark_columns[0]].mean() * 252 * 100)
        index_annual = float(index_returns.mean() * 252 * 100)
        reconciliation_gap = abs(first_cut_annual - index_annual)
        print(f"first cut's benchmark, mean daily return a year: {first_cut_annual:.2f} points")
        print(f"the KN US Equity Core's own, same window:       {index_annual:.2f} points")
        print(f"gap: {reconciliation_gap:.2f} points a year (tolerance 1.0)")
    else:
        reconciliation_gap = None
        print("the library returned no benchmark return column: the reconciliation cannot be read")

In [ ]:
# EXAMPLE-ONLY CELL
# Second layer: the factor model, on every book this experiment attributes.
if ATTRIBUTION_READY:
    decomposition = pandas.DataFrame({
        name: answer["factor_totals"]
        for name, answer in attributed.items()
        if "factor_totals" in answer
    })
    reserved = [
        "f_total_excess_returns",
        "f_total_factor_returns",
        "f_idyo_returns",
    ]
    priced_factors = [name for name in decomposition.index if name not in reserved]
    print("factor model on the rule, the control and equal weight, in points:")
    print(decomposition.loc[priced_factors, ["rule", "control", "equal weight"]]
          .round(2).sort_values("rule", ascending=False).to_string())
    print()
    print(decomposition.loc[[n for n in reserved if n in decomposition.index]].round(2).T
          .to_string())
    decomposition.to_csv(EXPERIMENT_DIRECTORY / "Attribution" / "factor_model.csv")

In [ ]:
# EXAMPLE-ONLY CELL
# Third pass: Brinson-Fachler on the residual, on the books attributed in full.
if ATTRIBUTION_READY:
    residual_table = pandas.DataFrame({
        name: answer["residual_totals"]
        for name, answer in attributed.items()
        if "residual_totals" in answer
    })
    print("Brinson-Fachler on the residual, in points:")
    print(residual_table.round(2).to_string())
    print()
    print("share of each book the factor model covers:", {
        name: round(answer["covered_share"], 3)
        for name, answer in attributed.items()
        if "covered_share" in answer
    })
    brinson_table = pandas.DataFrame({
        name: answer["brinson_totals"]
        for name, answer in attributed.items()
        if "brinson_totals" in answer
    })
    brinson_table.to_csv(EXPERIMENT_DIRECTORY / "Attribution" / "brinson_fachler.csv")
    residual_table.to_csv(EXPERIMENT_DIRECTORY / "Attribution" / "brinson_fachler_residual.csv")

## 6 · Counterfactuals — who earned the idiosyncratic share

The factor model leaves part of the book's excess return unexplained. That is a number, not an
answer: the book makes choices the index does not. **Each counterfactual removes exactly one of
those and keeps the rest**, which is the only way the question stops being an inference. The
engine can already price all of them; `AGENTS.md`, under *What attribution must report*, names four
follow-ups.

In [ ]:
# EXAMPLE-ONLY CELL
# The sweep, read as curves: each setting against its own control, and against the index.


def statistic(
    label: str,
    name: str,
) -> float:
    """One engine figure for one run."""
    return float(runs[label]["portfolio_stats"][STATISTICS[name]])


if ENGINE_READY:
    index_sharpe = float(runs["rule"]["benchmarks_stats"][backtest_engine.INDEX_IDENTIFIER][
        STATISTICS["Sharpe"]
    ])
    sweep_rows = {}

    for setting, _ in SWEEP:
        book_label = f"sweep, {setting}"
        control_label = f"sweep, {setting}, control"
        sweep_rows[setting] = {
            "Sharpe": statistic(book_label, "Sharpe"),
            "control Sharpe": statistic(control_label, "Sharpe"),
            "CAGR": statistic(book_label, "CAGR"),
            "control CAGR": statistic(control_label, "CAGR"),
            "index Sharpe": float(runs[book_label]["benchmarks_stats"][
                backtest_engine.INDEX_IDENTIFIER][STATISTICS["Sharpe"]]),
        }

    sweep = pandas.DataFrame(sweep_rows).T
    sweep["Sharpe margin"] = sweep["Sharpe"] - sweep["control Sharpe"]
    sweep["CAGR margin"] = sweep["CAGR"] - sweep["control CAGR"]
    sweep["keeps the sign on both"] = (sweep["Sharpe margin"] > 0) & (sweep["CAGR margin"] > 0)
    sweep["beats the index on Sharpe"] = sweep["Sharpe"] > sweep["index Sharpe"]
    print(sweep.round(4).to_string())
    sweep.to_csv(EXPERIMENT_DIRECTORY / "Backtest" / "sweep.csv")

    random_rows = {
        f"random {seed:02d}": {
            "Sharpe": statistic(f"random {seed:02d}", "Sharpe"),
            "CAGR": statistic(f"random {seed:02d}", "CAGR"),
        }
        for seed in RANDOM_SEEDS
    }
    random_table = pandas.DataFrame(random_rows).T

    if ATTRIBUTION_READY:
        random_table["idiosyncratic"] = [
            float(decomposition.loc["f_idyo_returns", label])
            if label in decomposition.columns else numpy.nan
            for label in random_table.index
        ]

    print()
    print(random_table.round(4).to_string())

## 7 · Verdict

**In words.** A notebook that ends in a number and no sentence gets read as whatever the reader
hoped.

Three sentences: does the book work; what attribution says about why; what the next experiment
should change. Then copy the numbers into [`FINDINGS_1.md`](FINDINGS_1.md) — **that file is the
record, this notebook is the method.**

If sections 4 and 5 reported "not installed", this notebook has produced a book and no result,
which is the honest outcome and not a failure.

## Handoff

| Output | Consumed by |
| --- | --- |
| `Portfolio/portfolio_weights.csv` | the backtest engine |
| `Portfolio/` — the readable book and the summaries | humans, and `FINDINGS_1.md` |
| `Backtest/` — the track record, and the book's daily weights | the attribution library, `FINDINGS_1.md`, and the comparison baseline for every later experiment |
| `Attribution/` | `FINDINGS_1.md`, and the comparison baseline for every later experiment |

Later experiments read the **same** panel, over the same window, with the same costs and the same
rebalancing convention, and change only the selection or the weighting — which is what makes the
comparison against Experiment 1 meaningful.

## Open items to carry forward

| # | Item | Why it matters |
| --- | --- | --- |
| 1 | **No result without the licensed engines.** | Without them the book is built but its performance is not measured, and the blueprint's return predictions stay open. |
| 2 | **Turnover is target-to-target, not realised.** | The realised figure is lower. The engine's own series is the one to quote. |
| 3 | **The attribution window is shorter than the backtest**, bound by the supplied files' coverage. | The two sets of numbers describe different periods. |
| 4 | **Delisting exits use one day of hindsight.** | Inert on a universe of live securities; load-bearing on any universe that retains delisted names. |
| 5 | **Cash is a real instrument**, so going flat costs commission and earns a yield. | A strategy that trades to cash often is partly a bet on the front end of the curve. Ask attribution about it. |
| 6 | **Every lever Experiment 1 declines** — a weight cap, a minimum holding count, risk-aware sizing — is a later experiment, and each has to beat this book to earn its place. | Complexity is added one lever at a time. |

In [ ]:
# EXAMPLE-ONLY CELL
# The gate's criteria 1 to 4 and the predictions, read from the numbers above, row by row. The
# verdict in words goes into FINDINGS_1.md; criterion 5 is a person's signature, never this cell's.
if ENGINE_READY:
    rule_sharpe = statistic("rule", "Sharpe")
    rule_cagr = statistic("rule", "CAGR")
    control_sharpe = statistic("control", "Sharpe")
    control_cagr = statistic("control", "CAGR")
    benchmark_sharpes = {
        identifier: float(statistics[STATISTICS["Sharpe"]])
        for identifier, statistics in runs["rule"]["benchmarks_stats"].items()
    }
    sub_period_rows = {}

    for period, _, _ in SUB_PERIODS:
        sub_period_rows[period] = {
            "rule Sharpe": statistic(f"rule, {period}", "Sharpe"),
            "control Sharpe": statistic(f"control, {period}", "Sharpe"),
            "rule CAGR": statistic(f"rule, {period}", "CAGR"),
            "control CAGR": statistic(f"control, {period}", "CAGR"),
        }

    sub_periods = pandas.DataFrame(sub_period_rows).T
    sub_periods["beats the control on both"] = (
        (sub_periods["rule Sharpe"] > sub_periods["control Sharpe"])
        & (sub_periods["rule CAGR"] > sub_periods["control CAGR"])
    )
    print(sub_periods.round(4).to_string())
    margins_met = (rule_sharpe - control_sharpe >= 0.03) and (rule_cagr - control_cagr >= 0.005)
    periods_met = int(sub_periods["beats the control on both"].sum()) >= 2
    benchmarks_met = all(rule_sharpe > sharpe for sharpe in benchmark_sharpes.values())
    criterion_1 = margins_met and periods_met and benchmarks_met
    criterion_3 = (
        int(sweep["keeps the sign on both"].sum()) >= 6
        and bool(sweep["beats the index on Sharpe"].all())
    )

    if ATTRIBUTION_READY:
        rule_idiosyncratic = float(decomposition.loc["f_idyo_returns", "rule"])
        random_beaten = int((random_table["idiosyncratic"] < rule_idiosyncratic).sum())
        residual_selection = float(residual_table.loc["selection", "rule"])
        brinson_selection = float(brinson_totals.get("selection", numpy.nan))
        reconciled = reconciliation_gap is not None and reconciliation_gap <= 1.0
        criterion_2 = (
            reconciled
            and brinson_selection > 0
            and rule_idiosyncratic > 0
            and residual_selection > 0
            and random_beaten >= 16
        )
    else:
        criterion_2 = False

    gate = pandas.DataFrame([
        {"criterion": "1, beats the benchmarks and its control", "passes": criterion_1,
         "evidence": f"Sharpe {rule_sharpe:.3f} against {benchmark_sharpes} and the control's "
                     f"{control_sharpe:.3f}; CAGR margin {(rule_cagr - control_cagr) * 100:+.2f} "
                     f"points; sub-periods beaten on both: "
                     f"{int(sub_periods['beats the control on both'].sum())} of 3"},
        {"criterion": "2, idiosyncratic alpha in both layers", "passes": criterion_2,
         "evidence": (f"reconciliation gap {reconciliation_gap}; first-cut selection "
                      f"{brinson_selection:.2f}; idiosyncratic {rule_idiosyncratic:.2f}; "
                      f"residual selection {residual_selection:.2f}; random books beaten "
                      f"{random_beaten} of 20")
         if ATTRIBUTION_READY else "attribution not run"},
        {"criterion": "3, survives perturbation", "passes": criterion_3,
         "evidence": f"{int(sweep['keeps the sign on both'].sum())} of 8 keep the sign on both; "
                     f"{int(sweep['beats the index on Sharpe'].sum())} of 8 beat the index"},
        {"criterion": "4, costs and capacity stated", "passes": True,
         "evidence": "the realistic row, turnover and the participation bound are printed above"},
    ])
    print()
    print(gate.to_string(index=False))
    predictions = pandas.Series({
        "1, the cross earns its keep": margins_met and periods_met,
        "2, a drawdown control": (
            statistic("rule", "max drawdown") > statistic("control", "max drawdown")
            and statistic("rule", "volatility") < statistic("control", "volatility")
        ),
        "3, traded value beats equal weight on Sharpe": (
            rule_sharpe > statistic("equal weight", "Sharpe")
        ),
    })
    print()
    print("predictions held:")
    print(predictions.to_string())
    gate.to_csv(EXPERIMENT_DIRECTORY / "Backtest" / "gate.csv", index=False)

## 8 · Verify

**Assertions that raise when this experiment's output is wrong.** Section 2.1 prints its
invariants; this section raises on them, and on what the notebook wrote, so a run that reaches the
last cell is one whose book and numbers hold. At the least:

- every invariant of section 2.1 holds;
- no name a blocking row of `Universe/Data_Issues.csv` lists is ever held, by the book, the
  control or any arm;
- `Portfolio/portfolio_weights.csv`, read back, has one column per rebalance date, each summing to
  one with the cash proxy, and no negative weight;
- **every engine run valued the window it was asked for**: the days it valued, counted against the
  trading days in that window — never against a fixed floor, which a run that stopped years early
  can clear. Nearly every trading day valued, and none missing from the window's end, where a
  truncated run loses them. Skipped, as section 4 is, when the engine is not installed;
- **the first cut's benchmark is the index**: its return, the mean day times 252, within a point
  a year of the index's own returns file over the same days. Wider, and the weights were paired
  with a return they already held, members went unpriced or the book was not widened, and no
  attribution figure stands until it closes;
- the attribution window lies inside the backtest's, and every arm has an idiosyncratic figure.
  Both are skipped, as section 5 is, when attribution did not run.

<!-- example: begin -->

Here the reconciliation is read where the first cut runs, and it is a condition of the gate's
criterion 2, which fails above a point a year; this notebook's Verify section does not raise on
it. On 2026-10-06 the gap was 0.54 points a year. Verify asserts the exclusions on the book; the
control and every arm are built from the same eligible set, which drops the excluded names in
section 1, and are not asserted on their own — that bullet was added after the record.

<!-- example: end -->

In [ ]:
# EXAMPLE-ONLY CELL
# The book: the invariants of section 2.1 raised; the weight files read back from disk; every
# engine run counted against the window it was asked for, or, without one, every weight file found
# in Portfolio/; the attribution's window and its arms.
VALUED_SHARE_REQUIRED = 0.99
UNVALUED_DAYS_AT_END_ALLOWED = 5
verifications = {
    f"invariant: {name}": passed
    for name, passed in invariants.items()
}
verifications["the window is the universe notebook's"] = (
    window_written["window_start"].iloc[0] == WINDOW_START.date().isoformat()
)
verifications["the exclusions are the blueprint's"] = EXCLUDED == EXPECTED_EXCLUSIONS
weight_file = pandas.read_csv(
    EXPERIMENT_DIRECTORY / "Portfolio" / "portfolio_weights.csv",
    index_col=0,
)
verifications["the weight file has one column per rebalance date"] = (
    weight_file.shape[1] == len(REBALANCE_DATES)
)
verifications["every weight-file column sums to one with the cash proxy"] = bool(
    ((weight_file.sum(axis=0) - 1.0).abs() <= 1e-4).all()
)
verifications["no weight in the file is negative"] = bool((weight_file >= 0).all().all())

if ENGINE_READY:
    tasks_by_name = {
        settings["portfolio_name"]: settings
        for settings in tasks
    }

    for label, run in runs.items():
        if not run["success"]:
            verifications[f"{label}: the engine priced it"] = False
            continue

        start = pandas.Timestamp(tasks_by_name[runs_by_label[label]]["start"])
        end = pandas.Timestamp(tasks_by_name[runs_by_label[label]]["end"])
        run_days = window_days[(window_days >= start) & (window_days <= end)]
        valued_days = pandas.DatetimeIndex(run["register"].index)
        valued_share = len(run_days.intersection(valued_days)) / len(run_days)
        unvalued_at_end = run_days[run_days > valued_days.max()]
        verifications[f"{label}: {valued_share:.1%} of its trading days valued"] = (
            valued_share >= VALUED_SHARE_REQUIRED
        )
        verifications[f"{label}: {len(unvalued_at_end)} trading days unvalued at the end"] = (
            len(unvalued_at_end) <= UNVALUED_DAYS_AT_END_ALLOWED
        )
else:
    verifications["every weight file the notebook wrote is in Portfolio/"] = all(
        (EXPERIMENT_DIRECTORY / "Portfolio" / f"{name}.csv").is_file()
        for name in runs_by_label.values()
    )

if ATTRIBUTION_READY:
    verifications["the attribution window lies inside the backtest's"] = bool(
        window.min() >= window_days.min() and window.max() <= window_days.max()
    )
    verifications["every attributed arm has an idiosyncratic figure"] = bool(
        decomposition.loc["f_idyo_returns"].notna().all()
    )

failed_checks = [
    name
    for name, passed in verifications.items()
    if not passed
]

if len(failed_checks) > 0:
    message = f"the experiment failed verification: {'; '.join(failed_checks)}"

    raise AssertionError(message)

engine_runs = len(runs) if ENGINE_READY else 0
print(f"verified: {len(verifications)} checks on the book, its files and {engine_runs} engine runs")